# 04b — Feature Extraction: Raw Audio (Waveform-Level)

| Field | Detail |
|---|---|
| **Notebook** | `04b_feature_extraction_raw_audio.ipynb` |
| **Pipeline stage** | H1 — Voice Emotion Recognition - 4 of 7 (Feature Extraction, raw-audio branch) |
| **Owner(s)** | *Thrithwaka Preethi Shakya* |
| **Created** | *28/08/2026* |
| **Last updated** | *28/08/2026* |
| **Upstream dependency** | `02_preprocessing.ipynb` (manifest + standardized audio). Independent of `04a` — either can run first. |
| **Downstream dependency** | `05a_model_cnn_raw.ipynb`, `05d_model_wav2vec2.ipynb` |
| **Research proposal reference** | Section 5.1 (Wav2Vec2 — comparison architecture) |
| **Literature reference** | Kilimci, Bayraktar & K\u00fc\u00e7\u00fckmanisa (2025), *Evaluating raw waveforms with deep learning frameworks for speech emotion recognition*, Multimedia Tools and Applications — source of the raw-CNN preprocessing convention replicated here, and the empirical finding (plain CNN beating CNN-LSTM on raw audio in 5 of 6 datasets) that motivated including a raw-audio CNN as one of this project's four H1 candidate models |

## Purpose

`04a_feature_extraction_handcrafted.ipynb` prepared engineered acoustic features (MFCC, pitch, energy, RMSE, ZCR, Chroma) for two of the four H1 candidate models. This notebook prepares the **other two candidates' inputs**, both of which operate directly on the raw waveform rather than any hand-crafted representation:

- **Raw-audio CNN** — a per-clip, z-score-normalized raw waveform, following the exact preprocessing convention described by Kilimci et al. (2025). This is the model that specific paper found consistently outperformed CNN-LSTM on raw audio across RAVDESS, SAVEE, and most of their other tested datasets — the direct empirical justification for including it as a fourth candidate architecture in this project's model-selection comparison (see `docs/research_proposal_mapping.md`).
- **Wav2Vec2** — the raw waveform processed through Hugging Face's official `Wav2Vec2FeatureExtractor`, which applies the exact normalization the pretrained `facebook/wav2vec2-base` checkpoint expects. We deliberately do **not** hand-roll this normalization ourselves — using the library's own extractor guarantees byte-for-byte compatibility with the pretrained model that `05d_model_wav2vec2.ipynb` will fine-tune.

**Why 16 kHz matters here specifically:** `02_preprocessing.ipynb` chose a 16,000 Hz standardization target *specifically* because Wav2Vec2 requires 16 kHz input natively. This notebook is where that decision pays off — no resampling is needed for either branch here, only normalization.

## Objectives

1. Load and validate the manifest produced by `02_preprocessing.ipynb`, and cross-check label encoding against `04a`'s saved metadata for consistency.
2. Extract and z-score-normalize raw waveforms for the raw-audio CNN candidate, following Kilimci et al. (2025)'s described preprocessing exactly.
3. Process raw waveforms through Wav2Vec2's official feature extractor for the Wav2Vec2 candidate, with a documented, tested fallback if Hugging Face Hub is unreachable.
4. Validate shape and value-range consistency for both branches.
5. Save both arrays, labels, and split assignment as one aligned artifact, plus full metadata.
6. Leave a structured handoff note for `05a` and `05d`.

## 0. Environment Setup

In [1]:
import sys
import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import librosa

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable, **kwargs):
        return iterable

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "config").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from config.settings import settings  # noqa: E402

print(f"Project root resolved to: {PROJECT_ROOT}")

Project root resolved to: C:\Users\thrit\Desktop\emotion-ai-companion-research


In [2]:
LOG_DIR = PROJECT_ROOT / "reports" / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = LOG_DIR / "04b_feature_extraction_raw_audio.log"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[logging.FileHandler(LOG_PATH, mode="w"), logging.StreamHandler(sys.stdout)],
)
logger = logging.getLogger("feature_extraction_raw_audio")
RANDOM_SEED = 42

## 1. Load & Validate Upstream Outputs

Same validation discipline as `04a`: standardization parameters are read from `02_preprocessing.ipynb`'s own report rather than re-hardcoded, so this notebook can never silently drift out of sync with the actual audio on disk.

In [3]:
preprocessing_report_path = PROJECT_ROOT / "reports" / "02_preprocessing_report.json"
if not preprocessing_report_path.exists():
    raise FileNotFoundError(f"{preprocessing_report_path} not found. Run 02_preprocessing.ipynb first.")

with open(preprocessing_report_path) as f:
    preprocessing_report = json.load(f)

assert preprocessing_report["class_count_validation_passed"], "Notebook 02 validation failed — fix before proceeding."
assert preprocessing_report["checksum_verification_passed"], "Notebook 02 checksum verification failed — fix before proceeding."

TARGET_SAMPLE_RATE = preprocessing_report["standardization"]["sample_rate_hz"]
TARGET_DURATION_SECONDS = preprocessing_report["standardization"]["duration_seconds"]
EXPECTED_N_SAMPLES = int(TARGET_SAMPLE_RATE * TARGET_DURATION_SECONDS)

assert TARGET_SAMPLE_RATE == 16000, (
    f"Standardized sample rate is {TARGET_SAMPLE_RATE} Hz, not 16000 Hz. "
    "Wav2Vec2 requires 16kHz input — this must be resolved in 02_preprocessing.ipynb, not worked "
    "around here, since a mismatch here would silently degrade Wav2Vec2 fine-tuning quality."
)

logger.info("Upstream validated. Standardized audio: %d Hz, %.1fs (%d samples/file).",
            TARGET_SAMPLE_RATE, TARGET_DURATION_SECONDS, EXPECTED_N_SAMPLES)
print(f"\u2705 Upstream validated. {TARGET_SAMPLE_RATE} Hz confirmed compatible with Wav2Vec2.")

2026-08-28 14:06:00,849 | INFO | Upstream validated. Standardized audio: 16000 Hz, 2.5s (40000 samples/file).
✅ Upstream validated. 16000 Hz confirmed compatible with Wav2Vec2.


In [4]:
manifest_path = PROJECT_ROOT / preprocessing_report["manifest_path"]
manifest_df = pd.read_csv(manifest_path)

# Defensive cross-platform path fix, same as 04a — see that notebook for full rationale.
for col in ["original_path", "standardized_path"]:
    manifest_df[col] = manifest_df[col].str.replace("\\\\", "/", regex=True)

print(f"Loaded manifest: {len(manifest_df)} rows")

Loaded manifest: 4068 rows


### Label encoding — reused, not redefined

Rather than reconstruct the label-to-integer mapping independently (risking a subtle inconsistency with `04a`), this notebook loads it directly from `04a`'s saved metadata and simply asserts it matches `config/settings.py` as a redundant safety check. Every H1 model must agree on identical integer labels for the model-selection comparison in `06_model_selection.ipynb` to be valid.

In [5]:
handcrafted_metadata_path = PROJECT_ROOT / "data" / "processed" / "features" / "h1_handcrafted_features_metadata.json"
if handcrafted_metadata_path.exists():
    with open(handcrafted_metadata_path) as f:
        handcrafted_metadata = json.load(f)
    label_to_idx = {k: int(v) for k, v in handcrafted_metadata["label_to_idx"].items()}
    logger.info("Reused label encoding from 04a's metadata.")
else:
    logger.warning(
        "04a's metadata not found — reconstructing label encoding from config/settings.py directly. "
        "Run 04a first if possible, so both notebooks are guaranteed to agree."
    )
    label_to_idx = {label: idx for idx, label in enumerate(settings.emotion_labels)}

idx_to_label = {idx: label for label, idx in label_to_idx.items()}
assert label_to_idx == {label: idx for idx, label in enumerate(settings.emotion_labels)}, (
    "Label encoding does not match config/settings.py — resolve before proceeding."
)
print("Label encoding (confirmed consistent with 04a and config/settings.py):")
for label, idx in label_to_idx.items():
    print(f"  {idx}: {label}")

2026-08-28 14:06:10,374 | INFO | Reused label encoding from 04a's metadata.
Label encoding (confirmed consistent with 04a and config/settings.py):
  0: happy
  1: sad
  2: angry
  3: fear
  4: neutral
  5: surprise


## 2. Branch 1 — Raw-Audio CNN Preprocessing

**Used by:** `05a_model_cnn_raw.ipynb`

Following Kilimci et al. (2025)'s described preprocessing exactly: *"raw sounds are first normalized to mean 0 and variance 1"*. Since every clip is already fixed-length (40,000 samples) from `02_preprocessing.ipynb`'s standardization, the clipping/padding step described in that paper is not needed here — our standardization notebook already guarantees uniform length by construction.

**Why per-clip normalization matters:** without it, a raw CNN would need to learn to be invariant to whatever loudness/recording-level differences exist between RAVDESS, TESS, and SAVEE's original recording setups — normalizing each clip to zero mean and unit variance removes that as a confound, letting the model focus on emotion-relevant waveform shape rather than incidental recording volume.

In [6]:
def normalize_raw_waveform(y: np.ndarray) -> np.ndarray:
    """Per-clip z-score normalization: mean 0, variance 1. Matches Kilimci et al. (2025)."""
    mean = y.mean()
    std = y.std()
    if std < 1e-8:  # guards against a near-silent/corrupted clip producing a division-by-zero
        logger.warning("Near-zero standard deviation encountered during normalization — returning zeroed clip.")
        return np.zeros_like(y, dtype=np.float32)
    return ((y - mean) / std).astype(np.float32)

## 3. Branch 2 — Wav2Vec2 Feature Extractor

**Used by:** `05d_model_wav2vec2.ipynb`

Uses Hugging Face's official `Wav2Vec2FeatureExtractor`, loaded from the `facebook/wav2vec2-base` checkpoint — the plain self-supervised pretrained model (not an ASR-fine-tuned variant like `wav2vec2-base-960h`), chosen deliberately as a clean starting point for fine-tuning on our own downstream task (emotion classification) without inheriting any ASR-specific bias.

Loading the feature extractor requires a one-time, lightweight download from the Hugging Face Hub (a small JSON config, not the full model weights). If no network access is available at extraction time, this section falls back to a manually-implemented normalization that matches the same default behavior (`do_normalize=True`: zero-mean, unit-variance per sample) — the fallback is logged explicitly so it's never silently used without the team knowing.

In [7]:
WAV2VEC2_CHECKPOINT = "facebook/wav2vec2-base"

try:
    from transformers import Wav2Vec2FeatureExtractor
    wav2vec2_extractor = Wav2Vec2FeatureExtractor.from_pretrained(WAV2VEC2_CHECKPOINT)
    USE_HF_EXTRACTOR = True
    logger.info("Loaded Wav2Vec2FeatureExtractor from '%s' via Hugging Face Hub.", WAV2VEC2_CHECKPOINT)
except Exception as exc:
    USE_HF_EXTRACTOR = False
    logger.warning(
        "Could not load Wav2Vec2FeatureExtractor from Hugging Face Hub (%s). "
        "Falling back to a manual zero-mean/unit-variance normalization matching the library's "
        "documented default behavior. Re-run this notebook with network access before final model "
        "training if possible, to guarantee exact reproducibility with the official extractor.",
        exc,
    )

print(f"Using official Hugging Face Wav2Vec2FeatureExtractor: {USE_HF_EXTRACTOR}")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

2026-08-28 14:06:35,032 | INFO | Loaded Wav2Vec2FeatureExtractor from 'facebook/wav2vec2-base' via Hugging Face Hub.
Using official Hugging Face Wav2Vec2FeatureExtractor: True


In [8]:
def extract_wav2vec2_input(y: np.ndarray, sr: int) -> np.ndarray:
    """Returns the normalized waveform in the exact format Wav2Vec2 expects."""
    if USE_HF_EXTRACTOR:
        processed = wav2vec2_extractor(
            y, sampling_rate=sr, return_tensors="np", padding=False
        )
        return processed["input_values"][0].astype(np.float32)
    else:
        # Documented fallback — matches Wav2Vec2FeatureExtractor's default do_normalize=True behavior.
        mean = y.mean()
        std = y.std()
        if std < 1e-8:
            return np.zeros_like(y, dtype=np.float32)
        return ((y - mean) / std).astype(np.float32)

## 4. Batch Extraction

In [9]:
raw_cnn_list, wav2vec2_list = [], []
failed_files = []

total = len(manifest_df)
for i, row in tqdm(manifest_df.iterrows(), total=total, desc="Extracting raw audio features"):
    audio_path = PROJECT_ROOT / row["standardized_path"]
    try:
        y, sr = librosa.load(str(audio_path), sr=None)
        assert sr == TARGET_SAMPLE_RATE, f"Unexpected sample rate {sr} (expected {TARGET_SAMPLE_RATE})"
        assert len(y) == EXPECTED_N_SAMPLES, f"Unexpected sample count {len(y)} (expected {EXPECTED_N_SAMPLES})"

        raw_cnn_list.append(normalize_raw_waveform(y))
        wav2vec2_list.append(extract_wav2vec2_input(y, sr))

    except Exception as exc:  # noqa: BLE001
        logger.error("Failed to extract raw-audio features for %s: %s", audio_path, exc)
        failed_files.append(str(audio_path))

    if (i + 1) % 500 == 0 or (i + 1) == total:
        logger.info("Processed %d / %d files", i + 1, total)

print(f"\nExtraction complete. Succeeded: {len(raw_cnn_list)} / {total}. Failed: {len(failed_files)}.")
if failed_files:
    print("\u26a0\ufe0f  Failed files (see log for full tracebacks):")
    for f in failed_files[:10]:
        print(f"   {f}")

Extracting raw audio features:   0%|          | 0/4068 [00:00<?, ?it/s]

2026-08-28 14:07:16,659 | INFO | Processed 500 / 4068 files
2026-08-28 14:07:21,772 | INFO | Processed 1000 / 4068 files
2026-08-28 14:07:26,860 | INFO | Processed 1500 / 4068 files
2026-08-28 14:07:31,801 | INFO | Processed 2000 / 4068 files
2026-08-28 14:07:37,060 | INFO | Processed 2500 / 4068 files
2026-08-28 14:07:42,150 | INFO | Processed 3000 / 4068 files
2026-08-28 14:07:47,233 | INFO | Processed 3500 / 4068 files
2026-08-28 14:07:53,005 | INFO | Processed 4000 / 4068 files
2026-08-28 14:07:53,839 | INFO | Processed 4068 / 4068 files

Extraction complete. Succeeded: 4068 / 4068. Failed: 0.


In [10]:
assert len(failed_files) == 0, (
    f"{len(failed_files)} files failed feature extraction — resolve before proceeding. See log: {LOG_PATH}"
)
manifest_df = manifest_df.iloc[: len(raw_cnn_list)].reset_index(drop=True)
print("\u2705 All files processed successfully with zero failures.")

✅ All files processed successfully with zero failures.


## 5. Shape and Value-Range Validation

Confirms both branches produced consistently-shaped output for every file, and sanity-checks that normalization actually did what it was supposed to (mean \u2248 0, std \u2248 1 per clip on average) — a quiet way to catch a normalization bug before it wastes hours of model-training compute downstream.

In [11]:
lengths_cnn = {arr.shape[0] for arr in raw_cnn_list}
assert len(lengths_cnn) == 1, f"Raw CNN branch has inconsistent lengths: {lengths_cnn}"
RAW_CNN_LENGTH = lengths_cnn.pop()
assert RAW_CNN_LENGTH == EXPECTED_N_SAMPLES
print(f"\u2705 Raw CNN branch: consistent length ({RAW_CNN_LENGTH} samples) across all {len(raw_cnn_list)} files.")

lengths_w2v = {arr.shape[0] for arr in wav2vec2_list}
assert len(lengths_w2v) == 1, f"Wav2Vec2 branch has inconsistent lengths: {lengths_w2v}"
WAV2VEC2_LENGTH = lengths_w2v.pop()
print(f"\u2705 Wav2Vec2 branch: consistent length ({WAV2VEC2_LENGTH} samples) across all {len(wav2vec2_list)} files.")

✅ Raw CNN branch: consistent length (40000 samples) across all 4068 files.
✅ Wav2Vec2 branch: consistent length (40000 samples) across all 4068 files.


In [12]:
raw_cnn_means = np.array([arr.mean() for arr in raw_cnn_list])
raw_cnn_stds = np.array([arr.std() for arr in raw_cnn_list])

print("Raw CNN branch normalization sanity check (should average close to mean=0, std=1):")
print(f"  Mean of per-clip means: {raw_cnn_means.mean():.6f} (expected \u2248 0)")
print(f"  Mean of per-clip stds:  {raw_cnn_stds.mean():.6f} (expected \u2248 1)")

near_zero_std_count = int((raw_cnn_stds < 0.01).sum())
print(f"\nClips with near-zero variance (likely silent/corrupted, normalized to all-zero): {near_zero_std_count}")
if near_zero_std_count > 0:
    logger.warning("%d clips had near-zero variance and were zero-filled instead of normalized.", near_zero_std_count)

Raw CNN branch normalization sanity check (should average close to mean=0, std=1):
  Mean of per-clip means: 0.000000 (expected ≈ 0)
  Mean of per-clip stds:  1.000000 (expected ≈ 1)

Clips with near-zero variance (likely silent/corrupted, normalized to all-zero): 0


## 6. Save Feature Arrays and Metadata

In [13]:
FEATURES_DIR = PROJECT_ROOT / "data" / "processed" / "features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_NPZ_PATH = FEATURES_DIR / "h1_raw_audio_features.npz"
OUTPUT_METADATA_PATH = FEATURES_DIR / "h1_raw_audio_features_metadata.json"

X_raw_cnn = np.stack(raw_cnn_list, axis=0)      # (N, 40000)
X_wav2vec2 = np.stack(wav2vec2_list, axis=0)     # (N, 40000)
y = manifest_df["label"].map(label_to_idx).values.astype(np.int64)
split = manifest_df["split"].values
source_dataset = manifest_df["source_dataset"].values

print(f"X_raw_cnn shape:  {X_raw_cnn.shape}")
print(f"X_wav2vec2 shape: {X_wav2vec2.shape}")

np.savez_compressed(
    OUTPUT_NPZ_PATH,
    X_raw_cnn=X_raw_cnn,
    X_wav2vec2=X_wav2vec2,
    y=y,
    split=split,
    source_dataset=source_dataset,
)
logger.info("Saved feature arrays to %s", OUTPUT_NPZ_PATH)
print(f"\nSaved: {OUTPUT_NPZ_PATH} ({OUTPUT_NPZ_PATH.stat().st_size / (1024**2):.1f} MB)")

X_raw_cnn shape:  (4068, 40000)
X_wav2vec2 shape: (4068, 40000)
2026-08-28 14:08:55,039 | INFO | Saved feature arrays to C:\Users\thrit\Desktop\emotion-ai-companion-research\data\processed\features\h1_raw_audio_features.npz

Saved: C:\Users\thrit\Desktop\emotion-ai-companion-research\data\processed\features\h1_raw_audio_features.npz (584.8 MB)


In [14]:
raw_audio_metadata = {
    "random_seed": RANDOM_SEED,
    "source_sample_rate_hz": TARGET_SAMPLE_RATE,
    "source_duration_seconds": TARGET_DURATION_SECONDS,
    "expected_n_samples": EXPECTED_N_SAMPLES,
    "raw_cnn_branch": {
        "description": "Per-clip z-score normalized raw waveform (mean 0, variance 1), matching Kilimci et al. (2025)",
        "used_by_model": "05a_model_cnn_raw.ipynb (plain CNN on raw audio)",
        "length": int(RAW_CNN_LENGTH),
        "normalization_check": {
            "mean_of_per_clip_means": float(raw_cnn_means.mean()),
            "mean_of_per_clip_stds": float(raw_cnn_stds.mean()),
            "near_zero_variance_clips": int(near_zero_std_count),
        },
    },
    "wav2vec2_branch": {
        "description": "Waveform processed via Wav2Vec2FeatureExtractor (or documented fallback)",
        "used_by_model": "05d_model_wav2vec2.ipynb (fine-tuned Wav2Vec2)",
        "checkpoint": WAV2VEC2_CHECKPOINT,
        "used_official_hf_extractor": bool(USE_HF_EXTRACTOR),
        "length": int(WAV2VEC2_LENGTH),
    },
    "label_to_idx": label_to_idx,
    "idx_to_label": idx_to_label,
    "total_files_processed": int(len(manifest_df)),
    "total_files_failed": int(len(failed_files)),
    "output_npz_path": str(OUTPUT_NPZ_PATH.relative_to(PROJECT_ROOT)),
}

with open(OUTPUT_METADATA_PATH, "w") as f:
    json.dump(raw_audio_metadata, f, indent=2)

logger.info("Saved feature metadata to %s", OUTPUT_METADATA_PATH)
print(f"Saved: {OUTPUT_METADATA_PATH}")
print(json.dumps(raw_audio_metadata, indent=2))

2026-08-28 14:09:01,454 | INFO | Saved feature metadata to C:\Users\thrit\Desktop\emotion-ai-companion-research\data\processed\features\h1_raw_audio_features_metadata.json
Saved: C:\Users\thrit\Desktop\emotion-ai-companion-research\data\processed\features\h1_raw_audio_features_metadata.json
{
  "random_seed": 42,
  "source_sample_rate_hz": 16000,
  "source_duration_seconds": 2.5,
  "expected_n_samples": 40000,
  "raw_cnn_branch": {
    "description": "Per-clip z-score normalized raw waveform (mean 0, variance 1), matching Kilimci et al. (2025)",
    "used_by_model": "05a_model_cnn_raw.ipynb (plain CNN on raw audio)",
    "length": 40000,
    "normalization_check": {
      "mean_of_per_clip_means": 1.9708219431424645e-11,
      "mean_of_per_clip_stds": 1.0,
      "near_zero_variance_clips": 0
    }
  },
  "wav2vec2_branch": {
    "description": "Waveform processed via Wav2Vec2FeatureExtractor (or documented fallback)",
    "used_by_model": "05d_model_wav2vec2.ipynb (fine-tuned Wav2Vec2)

## 7. Final Sanity Checks

In [15]:
reloaded = np.load(OUTPUT_NPZ_PATH, allow_pickle=True)

assert reloaded["X_raw_cnn"].shape == X_raw_cnn.shape
assert reloaded["X_wav2vec2"].shape == X_wav2vec2.shape
assert reloaded["y"].shape == y.shape
assert not np.isnan(reloaded["X_raw_cnn"]).any(), "NaN values remain in raw CNN branch!"
assert not np.isnan(reloaded["X_wav2vec2"]).any(), "NaN values remain in Wav2Vec2 branch!"
assert not np.isinf(reloaded["X_raw_cnn"]).any(), "Inf values found in raw CNN branch!"
assert not np.isinf(reloaded["X_wav2vec2"]).any(), "Inf values found in Wav2Vec2 branch!"

reloaded_label_counts = pd.Series(reloaded["y"]).map(idx_to_label).value_counts()
original_label_counts = manifest_df["label"].value_counts()
assert reloaded_label_counts.sort_index().equals(original_label_counts.sort_index()), (
    "Label distribution in saved features does not match the manifest!"
)

print("\u2705 Reloaded arrays match in-memory arrays exactly.")
print("\u2705 No NaN or Inf values in either branch.")
print("\u2705 Label distribution in saved file matches the source manifest exactly.")
print(f"\nSplit sizes in saved file: {pd.Series(reloaded['split']).value_counts().to_dict()}")

✅ Reloaded arrays match in-memory arrays exactly.
✅ No NaN or Inf values in either branch.
✅ Label distribution in saved file matches the source manifest exactly.

Split sizes in saved file: {'train': 3254, 'val': 407, 'test': 407}


---

## Summary Note — Handoff to Next Notebooks

*(Fill in the italicized placeholders after your first real run, before committing.)*

### What was done in this notebook

1. Validated `02_preprocessing.ipynb`'s report, confirming 16kHz standardization (a hard requirement for Wav2Vec2 compatibility).
2. Reused the exact label encoding from `04a_feature_extraction_handcrafted.ipynb` rather than redefining it independently.
3. Extracted **Raw CNN branch**: per-clip z-score normalized raw waveforms, `(N, 40000)`, following Kilimci et al. (2025)'s described preprocessing.
4. Extracted **Wav2Vec2 branch**: waveforms processed through the official `Wav2Vec2FeatureExtractor` for `facebook/wav2vec2-base` (or a documented, logged fallback if offline).
5. Verified zero extraction failures, consistent shapes, and sane normalization statistics (mean \u2248 0, std \u2248 1) for both branches.
6. Saved both arrays, labels, and split assignment into one aligned `.npz` file, plus full metadata.
7. Reloaded the saved file from disk and independently re-verified shape, NaN/Inf-freedom, and label-distribution correctness.

### Outputs produced by this notebook (and where to find them)

| Output | Location | Used by |
|---|---|---|
| Raw-audio feature arrays (both branches, labels, split) | `data/processed/features/h1_raw_audio_features.npz` | `05a_model_cnn_raw.ipynb` (reads `X_raw_cnn`), `05d_model_wav2vec2.ipynb` (reads `X_wav2vec2`) |
| Raw-audio feature metadata | `data/processed/features/h1_raw_audio_features_metadata.json` | Both training notebooks — confirms which Wav2Vec2 checkpoint/extractor path was actually used |
| Run log | `reports/logs/04b_feature_extraction_raw_audio.log` | Debugging any downstream shape-mismatch, or confirming whether the HF extractor fallback was triggered |

### How to load this notebook's output (for `05a`/`05d`)

```python
import numpy as np
import json

data = np.load("data/processed/features/h1_raw_audio_features.npz", allow_pickle=True)
X_raw_cnn, X_wav2vec2, y, split = data["X_raw_cnn"], data["X_wav2vec2"], data["y"], data["split"]

with open("data/processed/features/h1_raw_audio_features_metadata.json") as f:
    metadata = json.load(f)

X_train_raw = X_raw_cnn[split == "train"]
X_train_w2v = X_wav2vec2[split == "train"]
# ...same pattern for val/test
```

### What needs to be done next

1. **`04c_asr_transcription.ipynb`** (independent of this notebook — can run in parallel or has already run) — generates transcripts for the text-only DistilRoBERTa baseline.
2. **`05a_model_cnn_raw.ipynb`** — loads `X_raw_cnn`, applies the class weights already computed in `04a`'s metadata (reuse, don't recompute), trains the plain-CNN candidate.
3. **`05d_model_wav2vec2.ipynb`** — loads `X_wav2vec2`, fine-tunes `facebook/wav2vec2-base` for 6-class classification. **Important:** if this notebook's `used_official_hf_extractor` metadata field is `false` (the offline fallback was used), re-run this notebook with network access before final model training — training with the manual fallback and then fine-tuning the actual `Wav2Vec2ForSequenceClassification` model risks a subtle mismatch between how the input was normalized and what the pretrained model's internal layers expect.

**Before running `05a` or `05d`, confirm:**
- Section 5's shape-consistency checks both printed \u2705.
- Section 7's reload sanity checks all printed \u2705.
- `data/processed/features/h1_raw_audio_features_metadata.json`'s `total_files_failed` field reads `0`.
- If using `05d`: confirm `used_official_hf_extractor` is `true` in the metadata before trusting the Wav2Vec2 branch for final results.

### Resources needed for the next step

| Resource | Needed for | Notes |
|---|---|---|
| `data/processed/features/h1_raw_audio_features.npz` | Direct model input for `05a` and `05d` | Load once per notebook; do not re-extract independently |
| `data/processed/features/h1_raw_audio_features_metadata.json` | Confirms which normalization path was used for Wav2Vec2, and the exact checkpoint name to load for fine-tuning | Read `wav2vec2_branch.checkpoint` directly rather than hardcoding it again |
| `data/processed/features/h1_handcrafted_features_metadata.json` (from `04a`) | Class weights — reuse the same ones for consistency across all four H1 candidate models | Do not recompute separately per model notebook |
| `transformers` library (add to `requirements.txt` if not already present) | Loading `Wav2Vec2FeatureExtractor` here and `Wav2Vec2ForSequenceClassification` in `05d` | Network access needed on first use to download the pretrained checkpoint config/weights |

### Known issues / things to watch for

- *(Fill in anything discovered during this run — e.g. whether the Hugging Face Hub was reachable, or if any clips triggered the near-zero-variance fallback in Section 5.)*

### Run metadata

- **Run by:** *Thrithwaka Preethi Shakya*
- **Date:** *28/08/2026*
- **Files processed / failed:** *Succeeded: 4068 / 4068. Failed: 0.*
- **Official Wav2Vec2 extractor used:** *Yes*
- **Near-zero-variance clips found:** *0*